# In-Stream Processing

The previous examples demonstrated how to use concurrency to download data, which is useful for working offline or where Internet speeds are limited. Downloading data is useful for performing exploratory data analysis on a subset of data. 

Cloud resources support streaming data directly without the overhead of writing the file to disk. Streaming data is more efficient regarding total time to process data and is typically used when analyzing data sets that cover a large geographic area, a long period of time, or a combination of both.

## Streaming Data from Earthscope

This notebook demonstrates how to stream seismic miniseed files directly from FDSN servers and process them in parallel WITHOUT downloading first.

### Import packages

In [ ]:
import os
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed
from obspy import UTCDateTime
from obspy.clients.fdsn import Client
import io

### Function to Stream and Process a Single Station

In [ ]:
def stream_and_process_single_station(station, start, end, network="XD", location="*", channel="?HZ",
                                      starttime_override=None, endtime_override=None,
                                      output_dir="./imush_processed_data", fdsn_client="EARTHSCOPE",
                                      freqmin=0.1, freqmax=10.0,
                                      taper_percentage=0.05, corners=4, zerophase=True):
    """
    Stream miniseed data from FDSN server and process it in memory.
 
    This function combines downloading and processing into a single operation,
    streaming data directly from the server without saving raw files first.
 
    Each worker process creates its own FDSN client instance. This is required
    when using ProcessPoolExecutor because the Client object holds an SSL
    connection that cannot be pickled and sent between processes.
 
    Processing steps:
    1. Create a local FDSN client for this worker process
    2. Stream waveform data from FDSN server
    3. Linear detrend - removes linear trends in the data
    4. Demean - removes the mean value (centers data around zero)
    5. Taper - applies a window to reduce edge effects
    6. Bandpass filter - keeps only frequencies within specified range
    7. Save processed data to output directory
 
    Parameters:
    -----------
    station : str
        Station code (e.g., 'ANMO')
    start : str
        Station's start date from metadata
    end : str
        Station's end date from metadata
    network : str
        Network code (e.g., 'XD')
    location : str
        Location code (e.g., '*' for all locations)
    channel : str
        Channel code (e.g., 'BH?' for all BH channels)
    starttime_override : str or None
        Override start time if provided
    endtime_override : str or None
        Override end time if provided
    output_dir : str
        Directory to save the processed file
    fdsn_client : str
        FDSN client name (e.g., 'EARTHSCOPE', 'IRIS')
    freqmin : float
        Minimum frequency for bandpass filter (Hz)
    freqmax : float
        Maximum frequency for bandpass filter (Hz)
    taper_percentage : float
        Percentage of trace to taper (0-0.5)
    corners : int
        Number of corners for Butterworth filter
    zerophase : bool
        If True, apply zero-phase filter
 
    Returns:
    --------
    tuple : (success: bool, station: str, output_file: str or None, error: str or None)
    """
 
    actual_start = starttime_override if starttime_override is not None else start
    actual_end   = endtime_override   if endtime_override   is not None else end
    starttime = UTCDateTime(actual_start)
    endtime   = UTCDateTime(actual_end)
 
    try:
        # Step 1: Create a fresh FDSN client inside the worker process.
        # The client cannot be passed from the parent process because it holds
        # an SSL connection (SSLContext) that is not picklable. Creating it here
        # ensures each worker has its own independent connection.
        client = Client(fdsn_client)
 
        # Step 2: Stream waveform data directly from FDSN server into memory
        st = client.get_waveforms(
            network=network,
            station=station,
            location=location,
            channel=channel,
            starttime=starttime,
            endtime=endtime
        )
 
        n_traces = len(st)
 
        # Steps 3-6: Process each trace in memory
        for tr in st:
            tr.detrend('linear')
            tr.detrend('demean')
            tr.taper(max_percentage=taper_percentage, type='hann')
            tr.filter('bandpass',
                      freqmin=freqmin,
                      freqmax=freqmax,
                      corners=corners,
                      zerophase=zerophase)
 
        # Step 7: Save processed data to output directory
        filename = (f"{network}_{station}_"
                    f"{starttime.strftime('%Y%m%d')}_{endtime.strftime('%Y%m%d')}"
                    f"_processed.mseed")
        filepath = os.path.join(output_dir, filename)
        st.write(filepath, format='MSEED')
 
        print(f"✓ Streamed & processed: {station} ({n_traces} trace(s))")
        return (True, station, filepath, None)
 
    except Exception as e:
        error_msg = f"{type(e).__name__}: {str(e)}"
        print(f"✗ Failed: {station} - {error_msg}")
        return (False, station, None, error_msg)

>**Explainer:**
>
>This function combines the download and processing steps into a single operation that happens entirely in memory. Instead of downloading raw files first, it creates its own FDSN client and streams waveform data directly from the server using ObsPy's `get_waveforms()` method.
>
>Creating the client inside the worker function rather than passing one in from the parent is essential when using `ProcessPoolExecutor`. Because each worker runs in a separate Python process, any objects passed to it must be picklable — convertible to bytes for transmission across the process boundary. The FDSN client holds an SSL network connection internally which cannot be pickled, so each worker creates its own fresh connection independently instead.
>
>Once the data arrives, it is held in memory as an ObsPy Stream object and immediately processed with the same four-step workflow — detrend, demean, taper, filter. Only the final processed data is written to disk, which saves significant storage space and I/O time compared to saving raw files first. This approach is ideal for processing large datasets where storing both raw and processed files would be prohibitive, and because each worker manages its own client and data independently, multiple stations can be streamed and processed simultaneously without interference.

### Main Parallel Streaming and Processing Function

In [ ]:
def stream_and_process_parallel(station_rows, *, starttime=None, endtime=None,
                                output_dir="./stream_processed_data",
                                freqmin=0.1, freqmax=10.0,
                                taper_percentage=0.05, corners=4,
                                zerophase=True, max_workers=5,
                                network="XD", location="*", channel="HHZ",
                                fdsn_client="EARTHSCOPE"):
    """
    Stream and process miniseed data from FDSN servers in parallel.
 
    This function orchestrates parallel streaming and processing of seismic data
    from multiple stations using ProcessPoolExecutor. Data is streamed directly
    from FDSN servers and processed in memory without intermediate file storage.
 
    Unlike ThreadPoolExecutor, ProcessPoolExecutor spawns separate Python
    processes for each worker. This bypasses the GIL so CPU-bound tasks like
    filtering and detrending run truly in parallel. Because the FDSN client
    holds an SSL connection that cannot be pickled, it is NOT shared — each
    worker creates its own client independently.
 
    Parameters:
    -----------
    station_rows : iterable
        Iterable of station objects with .station, .start, .end attributes
        (e.g., rows from a pandas DataFrame)
    starttime : str, optional
        Override start time in format 'YYYY-MM-DD' or 'YYYY-MM-DDTHH:MM:SS'
    endtime : str, optional
        Override end time in format 'YYYY-MM-DD' or 'YYYY-MM-DDTHH:MM:SS'
    output_dir : str, optional
        Directory to save processed files (default: './stream_processed_data')
    freqmin : float, optional
        Minimum frequency for bandpass filter in Hz (default: 0.1)
    freqmax : float, optional
        Maximum frequency for bandpass filter in Hz (default: 10.0)
    taper_percentage : float, optional
        Percentage of trace to taper, 0-0.5 (default: 0.05 = 5%)
    corners : int, optional
        Number of corners for Butterworth filter (default: 4)
    zerophase : bool, optional
        If True, apply zero-phase filter (default: True)
    max_workers : int, optional
        Maximum number of parallel workers (default: 5)
    network : str, optional
        Network code (default: 'XD')
    location : str, optional
        Location code (default: '*')
    channel : str, optional
        Channel code (default: 'HHZ')
    fdsn_client : str, optional
        FDSN client name (default: 'EARTHSCOPE')
 
    Returns:
    --------
    dict : Dictionary containing 'successful' and 'failed' processing results
 
    Example:
    --------
    >>> import pandas as pd
    >>> stations_df = pd.DataFrame({
    ...     'station': ['ANMO', 'CCM', 'HLID'],
    ...     'start': ['2024-01-01', '2024-01-01', '2024-01-01'],
    ...     'end': ['2024-01-02', '2024-01-02', '2024-01-02']
    ... })
    >>> results = stream_and_process_parallel(
    ...     stations_df.itertuples(),
    ...     starttime='2024-01-01',
    ...     endtime='2024-01-02',
    ...     max_workers=5
    ... )
    """
 
    station_data = [(row.station, row.start, row.end) for row in station_rows]
 
    os.makedirs(output_dir, exist_ok=True)
 
    # The client is NOT created here. Each worker process creates its own
    # client inside stream_and_process_single_station to avoid the SSLContext
    # pickle error that occurs when passing a shared client to worker processes.
 
    print(f"{'='*70}")
    print(f"Stream & Process Seismic Data (In-Memory)")
    print(f"{'='*70}")
    print(f"FDSN server:      {fdsn_client}")
    print(f"Output directory: {output_dir}")
    print(f"Stations:         {len(station_data)}")
    print(f"Parallel workers: {max_workers}")
    print(f"\nData parameters:")
    print(f"  - Network:           {network}")
    print(f"  - Location:          {location}")
    print(f"  - Channel:           {channel}")
    if starttime:
        print(f"  - Start time:        {starttime}")
    if endtime:
        print(f"  - End time:          {endtime}")
    print(f"\nProcessing parameters:")
    print(f"  - Linear detrend:    enabled")
    print(f"  - Demean:            enabled")
    print(f"  - Taper:             {taper_percentage*100}% (Hann window)")
    print(f"  - Bandpass filter:   {freqmin}-{freqmax} Hz")
    print(f"  - Filter corners:    {corners}")
    print(f"  - Zero-phase:        {zerophase}")
    print(f"{'='*70}\n")
 
    results = {'successful': [], 'failed': []}
 
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        future_to_station = {
            executor.submit(
                stream_and_process_single_station,
                station, start, end, network, location, channel,
                starttime, endtime, output_dir, fdsn_client,
                freqmin, freqmax, taper_percentage, corners, zerophase
            ): station
            for station, start, end in station_data
        }
 
        for future in as_completed(future_to_station):
            station = future_to_station[future]
            try:
                success, station_name, output_file, error = future.result()
 
                if success:
                    results['successful'].append({
                        'station': station_name,
                        'output_file': output_file
                    })
                else:
                    results['failed'].append({
                        'station': station_name,
                        'error': error
                    })
            except Exception as e:
                error_msg = f"Unexpected error: {str(e)}"
                print(f"✗ Failed: {station} - {error_msg}")
                results['failed'].append({
                    'station': station,
                    'error': error_msg
                })
 
    print(f"\n{'='*70}")
    print(f"Processing Summary:")
    print(f"  ✓ Successful: {len(results['successful'])} stations")
    print(f"  ✗ Failed:     {len(results['failed'])} stations")
    print(f"{'='*70}")
 
    if results['failed']:
        print(f"\nFailed stations:")
        for item in results['failed']:
            print(f"  - {item['station']}: {item['error']}")
 
    return results

>**Explainer:**
>
>This is the main orchestration function that manages the parallel streaming and processing workflow. Unlike traditional approaches that download files first and then process them, this function combines both operations into a single streaming pipeline.
>
>It initializes an FDSN client that's shared across all worker processes, then uses `ProcessPoolExecutor` to simultaneously stream and process data from multiple stations. Each worker process runs in its own isolated Python interpreter, bypassing the GIL and allowing CPU-bound processing tasks like filtering and detrending to execute truly in parallel across CPU cores. Each worker independently connects to the FDSN server, streams waveform data into memory, processes it with the specified filters, and saves only the processed result — all without creating intermediate raw files.
>
>This approach offers several advantages:
>1. saves disk space by not storing raw data,
>2. reduces I/O overhead,
>3. allows processing to begin immediately as data arrives, and
>4. scales efficiently for large datasets by fully utilising multiple CPU cores rather than being limited by the GIL.
>
>The parallel execution means that if you're processing 20 stations with 5 workers, you'll get approximately 5x speedup compared to sequential processing, with the added benefit of reduced storage requirements.
>
>Note that because `ProcessPoolExecutor` uses separate processes rather than threads, any objects passed to worker functions — including the FDSN client — must be picklable, i.e., convert an object into a stream of bytes so it can be saved to disk or sent to another process. If the client cannot be pickled, each worker process should create its own client instance independently rather than sharing one.

### Memory Management

Jupyter notebooks running in a container have strict memory limits. If a process consumes more memory than available, the Python kernel will stop and restart. Multi-processing instantiates a Python interpreter for each worker. The result is multiplicative. For example, if a single task requires 0.5 Gb of memory, 10 workers will consume 5 Gb of memory. Working with data in-stream, also consumes a significant amount of memory. These can add up quickly and result in an out of memory error. The amount of available and used memory can be found at the bottom of the Jupyter Lab window.

![](./images/memory_usage.png)

The solution is to reduce the number of workers so that the memory required for multiple processes are within the allotted memory.

### Usage: Basic streaming and processing

In [ ]:
import pandas as pd

# Create a list of stations to process
stations = pd.DataFrame({
    'station': ['MG02', 'ME06', 'MG03', 'MF07'],
    'start': ['2014-06-01', '2014-06-01', '2014-06-01', '2014-06-01'],
    'end': ['2016-01-02', '2016-01-02', '2016-01-02', '2016-01-02']
})

# create directory for IMUSH data
# os.makedirs("stream_processed_data", exist_ok=True)

# Stream and process with default parameters
results = stream_and_process_parallel(
    stations.itertuples(),
    starttime='2014-07-17T00:00:00',
    endtime='2014-07-19T00:00:00',  # Process 2 days
    output_dir='./stream_processed_data',
    max_workers=10  # Process 10 stations simultaneously
)

> **Explainer:**
>
> This shows the simplest usage pattern where you define a list of stations (here using a pandas DataFrame) and call the main function with time boundaries. The function will stream data directly from the IRIS FDSN server for all stations in parallel, process each in memory, and save only the processed results.
>
> With 10 parallel workers, this can process 10 stations simultaneously, providing significant speedup for large station lists. The 12-hour time window is ideal for event-based studies where you want to capture a specific seismic event without downloading days of continuous data.

### Advanced Example with Custom Parameters

High-frequency processing for local events.

In [ ]:
# Process with parameters suitable for local/crustal events

# Read station list from the IMUSH.csv file
stations = pd.read_csv('IMUSH.csv')

results = stream_and_process_parallel(
    stations.itertuples(),
    starttime='2014-07-17',
    endtime='2014-07-24',  # One week of data
    output_dir='./weekly_imush_processed',
    freqmin=3.0,           # Higher frequency for local events
    freqmax=25.0,          # Capture higher frequencies
    taper_percentage=0.1,  # 10% taper
    corners=4,
    zerophase=True,
    max_workers=8,
    network='XD',
    channel='HHZ',         # Broadband high-gain channels
    fdsn_client='EARTHSCOPE'
)

print(f"\nProcessed {len(results['successful'])} stations successfully")
print(f"Failed to process {len(results['failed'])} stations")


> **Explainer:**
> 
> This example shows how to customize processing parameters for specific applications. The frequency range (5-10 Hz) is suited for local and crustal events, which have more high-frequency content than teleseismic events. Processing a full week of data demonstrates the efficiency advantage of streaming - you avoid storing 7 days × 4 stations = 28 raw files, instead only keeping the 4 processed files.
>
> The 10% taper (instead of the default 5%) provides more aggressive edge treatment, useful when you have concerns about boundary effects in the filtered data.

### Process Results and Quality Control

Analyzing and exporting results.

In [ ]:
# Display successful processing
print("\n" + "="*70)
print("Successfully Processed Stations:")
print("="*70)
for item in results['successful']:
    print(f"Station: {item['station']}")
    print(f"  Output: {item['output_file']}")
    print()

# Check for failures and investigate
if results['failed']:
    print("\n" + "="*70)
    print("Failed Stations (Need Attention):")
    print("="*70)
    for item in results['failed']:
        print(f"Station: {item['station']}")
        print(f"  Error: {item['error']}")
        print()

# Save results to CSV for documentation
successful_df = pd.DataFrame(results['successful'])
failed_df = pd.DataFrame(results['failed'])

if not successful_df.empty:
    successful_df.to_csv('streaming_successful.csv', index=False)
    print(f"✓ Saved successful results to: streaming_successful.csv")

if not failed_df.empty:
    failed_df.to_csv('streaming_failed.csv', index=False)
    print(f"✓ Saved failed results to: streaming_failed.csv")

# Calculate success rate
total = len(results['successful']) + len(results['failed'])
success_rate = len(results['successful']) / total * 100 if total > 0 else 0
print(f"\nSuccess rate: {success_rate:.1f}% ({len(results['successful'])}/{total})")

> **Explainer:**
> 
> This cell shows best practices for quality control and documentation after processing. It displays detailed information about both successful and failed stations, exports results to CSV files for record-keeping, and calculates a success rate metric. This
is particularly important when processing large numbers of stations, as it helps identify patterns in failures (e.g., certain stations might consistently fail due to data availability issues or network problems). The CSV files can be used for subsequent analysis or to retry failed stations with different parameters.

### Exercise

Try timing preprocessing downloaded miniseed files and compare it to preprocessing miniseed data in-stream. Which do you think would be faster at scale?

Start with reading 10 local miniseed files in the `serial_download` directory. The serial preprocessing function is provided. Fill in the blanks as needed. 

In [ ]:
# serial preprocessing skeleton

import time, os
from tqdm import tqdm
from pathlib import Path
from obspy import read

# preprocess a file
def preprocess(stream):
    
    # make a copy of the stream
    st_process = stream.copy()
    
    # detrend waveform
    st_process.detrend("linear")
    st_process.detrend("demean")    
    st_process.taper(max(10.0 / st[0].stats.sampling_rate, 0.05))
    st_process.filter("bandpass", freqmin=5, freqmax=10, corners=4, zerophase=True)

    return st_process
    

# Define the directory path
directory_path = Path("./serial_download") # Use Path('.') for the current directory

# Get a sorted list of all files and filter out directories immediately
all_files = sorted([p for p in directory_path.iterdir() if p.is_file()])

# Select the first 10 files
first_10_files = all_files[:10]

# create a directory to store preprocessed data
output_dir = "processed_data"
os.makedirs(output_dir, exist_ok=True)

# Time the download with tqdm
start_time = time.time()

with tqdm(total=1, desc="Downloading miniseed files", bar_format='{desc}: {elapsed}'):
    # write a loop that reads miniseed data and calls the preprocess function
    for file in _____:
        st = read(file)
        processed = preprocess(_____)
    
        # create file name to save processed stream
        file_name = "serial_processed_" + os.path.basename(file).split('/')[-1]
        filepath = os.path.join(output_dir, file_name)
        
        # write processed stream
        _____.write(filepath, format='MSEED')
        )

elapsed_time = time.time() - start_time
print(f"\nTotal download time: {elapsed_time:.2f} seconds ({elapsed_time/60:.2f} minutes)")
print(f"\nDownload time per station (10 stations): {elapsed_time/10:.2f}")

Next, preprocess the all the IMUSH data in-stream. Fill in the blanks.

In [ ]:
# Read station list from the IMUSH.csv file
stations = pd._____('IMUSH.csv')

with tqdm(total=1, desc="Downloading miniseed files", bar_format='{desc}: {elapsed}'):
    # use the stream_and_process_parallel parallel function (HINT: see basic usage)
    # write the processed files to new directory


elapsed_time = time.time() - start_time
print(f"\nTotal download time: {elapsed_time:.2f} seconds ({elapsed_time/60:.2f} minutes)")
print(f"\nDownload time per station (70 stations): {elapsed_time/70:.2f}")

## Summary Streaming vs Traditional Approach

**Key Advantages:**

1. *Reduced Storage*: Only processed data is saved, saving 50% disk space
2. *Faster Processing*: Download and processing happen simultaneously
3. *Better Scalability*: Easier to process hundreds or thousands of stations
4. *Simplified Workflow*: No intermediate file management needed
5. *Fresh Data*: Always gets the latest data from FDSN server

**Optimal Worker Count:**

- For network-bound operations: 5-20 workers work well
- Too many workers can overwhelm FDSN servers (be respectful!)
- Balance between speed and server load
- IRIS typically allows ~5-10 concurrent connections per user

**When to Use Streaming vs Traditional:**

Use Streaming when:
- Processing many stations with limited disk space
- Need only processed data, not raw data
- Working with cloud computing (minimize storage costs)

Use Traditional (download first) when:
- Need to preserve raw data for multiple processing runs
- Working offline or with unreliable network
- Need to experiment with different processing parameters
- Archiving data for long-term studies